In [1]:
import functions as f
import pandas as pd
import numpy as np

from 'function' I will use add_combinations and add_prev_days

I will check the model for following combinations:
- original data
- standarized
- without outliers
- with combinations
- with prev days
- with columns chosen only from random forest

# Upload data

In [4]:
X_train = pd.read_csv('data/to_model/X_train.csv')
X_valid = pd.read_csv('data/to_model/X_val.csv')
y_train = pd.read_csv('data/to_model/Y_train.csv')
y_valid = pd.read_csv('data/to_model/Y_val.csv')
X_train.head()

,pressure,maxtemp,temparature,mintemp,dewpoint,humidity,cloud,sunshine,windspeed,day_repaired_sin,day_repaired_cos,winddirection_sin,winddirection_cos
0,1017.4,21.2,20.6,19.9,19.4,87.0,88.0,1.1,17.2,0.000000,1.000000,0.951057,0.309017
1,1019.5,16.2,16.9,15.8,15.4,95.0,91.0,0.0,21.9,0.017261,0.999851,0.866025,0.500000
2,1024.1,19.4,16.1,14.6,9.3,75.0,47.0,8.3,18.1,0.034516,0.999404,0.994522,0.104528
3,1013.4,18.1,17.8,16.9,16.8,95.0,95.0,0.0,35.6,0.051761,0.998659,0.951057,0.309017
4,1021.8,21.3,18.4,15.2,9.6,52.0,45.0,3.6,24.8,0.068991,0.997617,0.743145,0.669131


In [ ]:
test = pd.read_csv('data/test.csv')
X_test = test.drop(columns=['rainfall'])

# Make different data combinations

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

In [7]:
def standarize_data(data):
    scaler = StandardScaler()
    data_standardScaler = scaler.fit_transform(data)
    return pd.DataFrame(data_standardScaler, columns=data.columns)

In [21]:
def remove_outliers(X, y):
    iso_forest = IsolationForest(contamination=0.01, random_state=42)
    predictions = iso_forest.fit_predict(X)
    mask = predictions != -1
    X_cleaned = X[mask].reset_index(drop=True)
    y_cleaned = y[mask].reset_index(drop=True)
    return X_cleaned, y_cleaned

In [23]:
X_train_standard = standarize_data(X_train)
X_valid_standard = standarize_data(X_valid)

X_train_combinations = f.add_combinations(X_train)
X_valid_combinations = f.add_combinations(X_valid)

X_train_prev_days = f.add_prev_days(X_train)
X_valid_prev_days = f.add_prev_days(X_valid)

X_train_noOutliers, y_train_noOutliers = remove_outliers(X_train, y_train)
X_valid_noOutliers, y_valid_noOutliers = remove_outliers(X_valid, y_valid)

# Evaluation
all models will be evaluated on ROC AUC score

In [10]:
from sklearn.metrics import roc_auc_score
def evaluate_model(model, X_train, X_valid, y_train, y_valid):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)
    print(roc_auc_score(y_valid, y_pred))
    return roc_auc_score(y_valid, y_pred)

# Model CatBoost

In [11]:
from catboost import CatBoostClassifier

In [19]:
categorical_features_indices = [i for i, col in enumerate(X_train.columns) if X_train[col].dtype == 'object']

model = CatBoostClassifier(
    iterations=400,
    learning_rate=0.05,
    depth=6,
    cat_features=categorical_features_indices,
    verbose=50
)

evaluate_model(model, X_train, X_valid, y_train, y_valid)

0:	learn: 0.6530644	total: 5.49ms	remaining: 2.19s
50:	learn: 0.2921229	total: 255ms	remaining: 1.75s
100:	learn: 0.2542611	total: 472ms	remaining: 1.4s
150:	learn: 0.2251641	total: 692ms	remaining: 1.14s
200:	learn: 0.1916726	total: 883ms	remaining: 874ms
250:	learn: 0.1678826	total: 1.07s	remaining: 638ms
300:	learn: 0.1476951	total: 1.29s	remaining: 425ms
350:	learn: 0.1287129	total: 1.48s	remaining: 207ms
399:	learn: 0.1137332	total: 1.67s	remaining: 0us
0.7584506157325961


0.7584506157325961

In [13]:
evaluate_model(model, X_train_standard, X_valid_standard, y_train, y_valid)

0:	learn: 0.6531839	total: 6.17ms	remaining: 2.46s
50:	learn: 0.2919319	total: 154ms	remaining: 1.05s
100:	learn: 0.2548091	total: 277ms	remaining: 820ms
150:	learn: 0.2251998	total: 417ms	remaining: 688ms
200:	learn: 0.1928949	total: 579ms	remaining: 573ms
250:	learn: 0.1674623	total: 701ms	remaining: 416ms
300:	learn: 0.1447607	total: 821ms	remaining: 270ms
350:	learn: 0.1266974	total: 938ms	remaining: 131ms
399:	learn: 0.1113861	total: 1.06s	remaining: 0us
0.7555604423221914


0.7555604423221914

In [15]:
evaluate_model(model, X_train_combinations, X_valid_combinations, y_train, y_valid)

0:	learn: 0.6502902	total: 69.5ms	remaining: 27.7s
50:	learn: 0.2726635	total: 3.77s	remaining: 25.8s
100:	learn: 0.2209697	total: 7.42s	remaining: 22s
150:	learn: 0.1877114	total: 11.3s	remaining: 18.7s
200:	learn: 0.1584922	total: 15.2s	remaining: 15.1s
250:	learn: 0.1323744	total: 18.1s	remaining: 10.7s
300:	learn: 0.1107060	total: 20.1s	remaining: 6.62s
350:	learn: 0.0926174	total: 22.3s	remaining: 3.11s
399:	learn: 0.0794492	total: 24.6s	remaining: 0us
0.7685662226690123


0.7685662226690123

In [20]:
evaluate_model(model, X_train_noOutliers, X_valid_noOutliers, y_train, y_valid)

CatBoostError: Length of label=1752 and length of data=1734 is different.